# 00 快速上手：把激活引导搬进 vLLM

**这篇写给谁**：你会用 PyTorch 和 Hugging Face 做实验，想把一个改变推理行为的算法放进真实的
vLLM 服务里验证。我们用一个最容易理解的算法开头：**激活引导**（activation steering）。

激活引导的做法很简单（ActAdd，Turner et al. 2023；CAA，Rimsky et al. 2024）：拿几对意思相反的
句子，比如"我爱……"和"我恨……"，在某一层取它们的激活之差，平均得到一个方向向量；生成时把这个
向量乘上一个强度，加到这一层的输出上。模型的语气就会朝这个方向偏。

在 Hugging Face 里，这是一个五行的 forward hook。到了 vLLM 就不一样了：模型代码被编译、被 CUDA
graph 捕获，几十个请求的 token 被拼成一个张量连续批处理，hook 既不知道哪一行属于哪个请求，也
可能根本不会被调用。你会想去改 vLLM 的模型代码，这正是 pllm 想让你省掉的事。

这篇教程走五步：

1. 用 pllm 起一个普通的 vLLM 引擎，确认环境没问题；
2. 用你熟悉的 Hugging Face 代码算出引导向量；
3. 写一个负载（workload）：两行声明；
4. 把向量送进引擎，调强度，看输出怎么变；
5. 回头看框架替你做了什么。

全程用 Qwen2.5-0.5B-Instruct，一张消费级 GPU 足够。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import SamplingParams

import pllm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# 一个小引擎：最多 4 个并发请求，只占 30% 显存，给同一张卡上的 Hugging Face 模型留出空间。
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write one sentence about my new job.", "Describe your weekend in one sentence."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=40, temperature=0.0)

## 第一步：不带负载，就是普通的 vLLM

`pllm.make_llm(config)` 不传 `workload` 时，起的就是一个不带负载的普通 vLLM 引擎，用法和 vLLM 的 `LLM.generate` 一样。先用它生成一次，
记下模型原本的回答，后面拿来对照。

In [3]:
plain = pllm.make_llm(CONFIG)
baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]
for text in baseline:
    print(repr(text))

del plain  # 释放显存，给后面的 Hugging Face 模型和新引擎腾地方
gc.collect()
torch.cuda.empty_cache()

'As an AI language model, I am here to assist you with your new job and provide you with the best possible assistance possible.'
"As an AI language model, I don't have a physical presence, so I don't have a weekend. However, I'm always here to assist you with any questions or tasks you may have!"


## 第二步：用你自己的代码算引导向量

这一步和 pllm 无关，就是你平时的研究代码。我们在第 12 层取五对反义短语最后一个 token 的激活，
求差再平均。Hugging Face 的 `hidden_states[12 + 1]` 是第 12 层输出之后的残差流。

In [4]:
LAYER = 12
pairs = [("I love", "I hate"), ("This is wonderful", "This is terrible"), ("I feel happy", "I feel sad"),
         ("What a great day", "What an awful day"), ("Everything is amazing", "Everything is horrible")]

hf = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).cuda()


@torch.no_grad()
def last_token_activation(text: str) -> torch.Tensor:
    ids = tokenizer(text, return_tensors="pt").to("cuda")
    return hf(**ids, output_hidden_states=True).hidden_states[LAYER + 1][0, -1].float()


vector = torch.stack([last_token_activation(a) - last_token_activation(b) for a, b in pairs]).mean(0)
print("引导向量:", tuple(vector.shape), "范数", round(vector.norm().item(), 2))

del hf
gc.collect()
torch.cuda.empty_cache()

引导向量: (896,) 范数 7.08


## 第三步：写负载

一个负载是 `pllm.Workload` 的子类，里面只放**声明**和你的张量数学。激活引导连数学都不用写，
两行声明就够：

- `steer` 是一份**状态**：一块由框架管理的显存，这里只有一个字段 `vector`，长度等于模型的
  hidden 宽度（`pllm.Field` 的形状函数在加载模型后才被调用，所以不用写死 896）。
  `imported=True` 表示它的值由你从外面送进来，而不是由负载自己训练。
- `patch` 是一个**激活补丁**：把 `steer` 加到第 12 层的输出上。`operator="add"` 表示直接相加；
  `decode=True` 表示每个新生成的 token 都加（默认还会加在 prompt 的最后两个 token 上）。

vLLM 的每一层返回"这一层新算出的部分"和"残差"两半，下一层开头才把它们相加，所以加到层输出上
和 Hugging Face 里加到 `hidden_states[13]` 上是同一件事。

In [5]:
class Steering(pllm.Workload):
    steer = pllm.InstanceState({"vector": pllm.Field(lambda width, _: (width,))}, imported=True)
    patch = pllm.model.ActivationPatch("steer", layers=(12,), operator="add", decode=True)

## 第四步：送进向量，调强度

把负载交给 `make_llm`，就得到一个带负载的引擎。向量通过 `generate` 的 `batch=` 参数送进去：
`pllm.io.StateImport` 按字段顺序装着要写入状态的张量。每次带 `batch=` 的调用都会发布状态的一个
新版本，这次调用里的请求读到的就是它；不带 `batch=` 的调用沿用上一次的值。

我们试三个强度：0（应当和原始输出完全一样）、+2（往"正面"推）、-2（往"负面"推）。

In [6]:
with pllm.make_llm(CONFIG, workload=Steering()) as llm:
    for strength in (0.0, 2.0, -2.0):
        outputs = llm.generate(prompts, greedy, batch=pllm.io.StateImport((strength * vector,)))
        texts = [output.outputs[0].text for output in outputs]
        print(f"强度 {strength:+.0f}:")
        for text in texts:
            print("   ", repr(text))
        if strength == 0.0:
            print("    与原始 vLLM 输出相同:", texts == baseline)
    print("eager 原因:", llm.diagnostics().eager_reason)

强度 +0:
    'As an AI language model, I am here to assist you with your queries and provide you with information and insights on various topics. I am always here to help you with your new job.'
    "As an AI language model, I don't have a physical presence, so I don't have a weekend. However, I'm always available to assist you with any questions or tasks you may have!"
    与原始 vLLM 输出相同: False
强度 +2:
    'As a new member of the team, I am excited to share that the journey of a new career at Alibaba Cloud, a leader in the digital transformation of the world, is a journey that sets the'
    'As a Qwen, I am here to share the unique experiences of my journey through the world, from the stars in the night sky, the sounds of nature, and the stories that connect us in'
强度 -2:
    "I'm not sure I understand what job I'm actually doing. I'm not even sure I know what job I'm actually doing. I'm not even sure I even know what job I should be"
    'I have no idea what I should do on my weekend, I 

强度 +2 时回答变得热情起来（"I am excited to share..."），-2 时变得迷茫、消极（"I have no idea what I
should do..."）。0.5B 的小模型被推得太猛会开始重复，这是激活引导本身的特性：强度和层号是你要调的超参数。

强度 0 的输出和第一步的原生 vLLM 只在措辞上有出入。原因不在补丁的内容（加的是零向量），而在执行方式：
最后一行说明，激活补丁目前不在 CUDA graph 里运行，pllm 自动改用 eager 模式，原因写在
`diagnostics().eager_reason` 里。eager 模式不走 CUDA graph 和 torch.compile，用的是另一套算子实现，bf16
的舍入略有不同，贪心解码在几乎打平的候选词之间就可能选到另一个。下一篇的观测负载不需要这个代价，
输出会和原生 vLLM 逐字相同。

## 第五步：框架替你做了什么

回头看，你写的只有两行声明和一次 `StateImport`。下面这些事是 pllm 做的：

- **找到这一层、改对这些行**：补丁装在 vLLM 编译后的第 12 层上，只作用于每个请求该改的 token，
  不管它们在连续批处理里被拼到了张量的哪一行；
- **管理这块状态**：`steer` 的显存在启动时按预算分配；每次 `StateImport` 产生一个新版本，
  已经在生成中的请求继续用它开始时的版本，不会在一句话中途被换掉；
- **不碰其他东西**：补丁只作用在你声明的那一层、那些 token 上，其余计算照旧。

## 下一步

- 想用训练好的低秩干预（ReFT / LoReFT），把 `operator` 换成 `"low_rank_affine"`，现成实现在
  `pllm.workloads.ReFTWorkload`；
- [01 观测](01_observe_probe.ipynb)：不改模型，只把每个 token 的内部状态读出来，用探针实时监控生成。